**Rules for the code:**

- Include all the code you used for your report in this file. The code for any section in the report should go under the same section in this file.
- Any missing code will result in -20% from its corresponding section in the report.
- Any irrelevant code will result in -20% from its corresponding section in the report.
- Make sure that you run your code before rendering so that all the necessary visual/numeric outputs are visible.
- Any code that is not properly run or throws errors will be considered missing/irrelevant.

In [1]:
import pandas as pd
import numpy as np

## 4) Data

Data Cleaning Portion

In [2]:
#Crash data cleaning
crash = pd.read_csv('Traffic_Crashes_-_Crashes_20251109.csv')


/var/folders/_l/b38cj6s513957kzfnj_20g5m0000gn/T/ipykernel_58949/2072339022.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  crash = pd.read_csv('Traffic_Crashes_-_Crashes_20251109.csv')


In [3]:
injury_columns = [
    'INJURIES_FATAL',
    'INJURIES_INCAPACITATING',
    'INJURIES_NON_INCAPACITATING',
    'INJURIES_REPORTED_NOT_EVIDENT',
    'INJURIES_NO_INDICATION',
    'INJURIES_UNKNOWN'
]
negative_value_check = {}
for column in injury_columns:
    has_negative = (crash[column]<0).any()
    negative_value_check[column] = has_negative
check_results = pd.Series(negative_value_check)

print("Presence of negative values")
print(check_results)

Presence of negative values
INJURIES_FATAL                   False
INJURIES_INCAPACITATING          False
INJURIES_NON_INCAPACITATING      False
INJURIES_REPORTED_NOT_EVIDENT    False
INJURIES_NO_INDICATION           False
INJURIES_UNKNOWN                 False
dtype: bool


In [4]:
empty_value_count = {}
for column in injury_columns:
    count = crash[column].isnull().sum()
    empty_value_count[column] = count
check_results = pd.Series(empty_value_count)
print("Number of empty values")
print(check_results)

crash_injury_filtered = crash.dropna(subset=injury_columns)
crash_injury_filtered = crash_injury_filtered.dropna(subset = ['BEAT_OF_OCCURRENCE'])

Number of empty values
INJURIES_FATAL                   2171
INJURIES_INCAPACITATING          2171
INJURIES_NON_INCAPACITATING      2171
INJURIES_REPORTED_NOT_EVIDENT    2171
INJURIES_NO_INDICATION           2171
INJURIES_UNKNOWN                 2171
dtype: int64


In [5]:
#merging in beat population data
beat_data = pd.read_csv('beatpop.txt', sep='\s+').drop(columns = ['area'])
beat_data.head()

,beat_number,population
0,111,1982.585454
1,112,1075.282355
2,113,1072.951867
3,114,16145.434610
4,121,6400.496775


In [6]:
crash_injury_filtered['BEAT_OF_OCCURRENCE'] = crash_injury_filtered['BEAT_OF_OCCURRENCE'].astype(int)
crash_merged = crash_injury_filtered.merge(beat_data, left_on='BEAT_OF_OCCURRENCE', right_on='beat_number', how='inner')
crash_merged.shape

(995101, 50)

In [7]:
#weather cleaning and merge into crash database
weather = pd.read_csv('KMDW.csv')
weather['DateTime_String'] = weather['Date'].astype(str) + ' ' + weather['Time'].astype(str)
weather['US_Time'] = pd.to_datetime(weather['DateTime_String'], format = '%Y-%m-%d %I:%M %p')
crash_merged['US_Time'] = pd.to_datetime(
    crash_merged['CRASH_DATE'], 
    format='%m/%d/%Y %I:%M:%S %p'
)
weather = weather.sort_values(by = 'US_Time', ascending = True)
crash_merged_sorted = crash_merged.sort_values(by = 'US_Time', ascending = True)


In [23]:
from datetime import timedelta
crash_merged_with_weather = pd.merge_asof(
    crash_merged_sorted, 
    weather, 
    on='US_Time', 
    direction='nearest', 
    tolerance=timedelta(minutes=120)
)

In [44]:
crash_merged_with_weather['INJURY_SCORE'] = crash_merged_with_weather['INJURIES_FATAL']*5 + crash_merged_with_weather['INJURIES_INCAPACITATING']*3 + crash_merged_with_weather['INJURIES_NON_INCAPACITATING']*2 + \
crash_merged_with_weather['INJURIES_REPORTED_NOT_EVIDENT']*1 + crash_merged_with_weather['INJURIES_UNKNOWN']*1 +crash_merged_with_weather['INJURIES_NO_INDICATION']*0.5

In [29]:
def time_of_day(hour):
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 18:
        return 'Afternoon'
    elif 18 <= hour < 22:
        return 'Evening'
    else:
        return 'Night'

crash_merged_with_weather['TIME_OF_DAY'] = crash_merged_with_weather['CRASH_HOUR'].apply(time_of_day)
predictors = ['US_Time', 'NUM_UNITS', 'population', 'TIME_OF_DAY', 'POSTED_SPEED_LIMIT', 'Precipitation Rate in mm', 'LIGHTING_CONDITION', 'Temperature_C', 'Speed in km/h']

crash_merged_with_weather[predictors].isnull().sum()


US_Time                          0
NUM_UNITS                        0
population                       0
TIME_OF_DAY                      0
POSTED_SPEED_LIMIT               0
Precipitation Rate in mm    118859
LIGHTING_CONDITION               0
Temperature_C               118859
Speed in km/h               118859
dtype: int64

In [39]:
#imputing in values for precip, temp and windspeed with ffill
crash_predictors_with_time = crash_merged_with_weather[predictors]
crash_predictors_with_time = crash_predictors_with_time.sort_values('US_Time')

for col in ['Precipitation Rate in mm', 'Temperature_C', 'Speed in km/h']:
    crash_predictors_with_time[col] = crash_predictors_with_time[col].ffill()



In [40]:
crash_predictors_with_time.isnull().sum()

US_Time                     0
NUM_UNITS                   0
population                  0
TIME_OF_DAY                 0
POSTED_SPEED_LIMIT          0
Precipitation Rate in mm    0
LIGHTING_CONDITION          0
Temperature_C               0
Speed in km/h               0
dtype: int64

## 5) Prediction

In [75]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
#redefine predictors here without us_time
predictors = ['NUM_UNITS', 'TIME_OF_DAY', 'POSTED_SPEED_LIMIT', 'Precipitation Rate in mm', 'population', 'LIGHTING_CONDITION', 'Temperature_C', 'Speed in km/h']


X = crash_predictors_with_time[predictors]
X = X.rename(columns={'NUM_UNITS': '#_of_vehicles', 'population': 'beat_population', 'Speed in km/h':'Wind speed(km/h)'})

X_ohe = pd.get_dummies(X)



In [76]:

y = crash_merged_with_weather['INJURY_SCORE']

X_train, X_test, y_train, y_test = train_test_split(X_ohe, y, test_size=0.1, random_state=1)


In [77]:
from sklearn.metrics import r2_score

results = []
col_groups = []
i = 0

while i < len(X_ohe.columns):
    col = X_ohe.columns[i]
    base_name = col.rsplit('_', 1)[0]
    group = [c for c in X_ohe.columns[i:] if c.startswith(base_name + '_')]
    
    if len(group) > 1:
        col_groups.append(group)
        i += len(group)
    else:
        col_groups.append([col])
        i += 1

current_cols = []
for group in col_groups:
    current_cols.extend(group)
    
    model = LinearRegression()
    model.fit(X_train[current_cols], y_train)
    
    train_pred = model.predict(X_train[current_cols])
    test_pred = model.predict(X_test[current_cols])
    
    results.append({
        'Predictors': ', '.join(current_cols),
        'Train_RMSE': np.sqrt(mean_squared_error(y_train, train_pred)),
        'Test_RMSE': np.sqrt(mean_squared_error(y_test, test_pred)),
        'Train_MAE': mean_absolute_error(y_train, train_pred),
        'Test_MAE': mean_absolute_error(y_test, test_pred),
        'Train_R2': r2_score(y_train, train_pred),
        'Test_R2': r2_score(y_test, test_pred)
    })

results_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)

results_df

,Predictors,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Train_R2,Test_R2
0,#_of_vehicles,1.113769,1.115746,0.687197,0.684028,0.027860,0.025276
1,"#_of_vehicles, POSTED_SPEED_LIMIT",1.107883,1.110518,0.680936,0.678076,0.038107,0.034390
2,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm",1.107881,1.110508,0.680931,0.678061,0.038111,0.034407
3,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm, beat_population",1.106676,1.109251,0.679458,0.676353,0.040202,0.036591
4,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm, beat_population, Temperature_C",1.106335,1.109047,0.679058,0.676032,0.040794,0.036946
5,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm, beat_population, Temperature_C, Wind speed(km/h)",1.106334,1.109050,0.679062,0.676037,0.040795,0.036942
6,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm, beat_population, Temperature_C, Wind speed(km/h), TIME_OF_DAY_Afternoon, TIME_OF_DAY_Evening, TIME_OF_DAY_Morning, TIME_OF_DAY_Night",1.105557,1.108538,0.677606,0.674808,0.042142,0.037830
7,"#_of_vehicles, POSTED_SPEED_LIMIT, Precipitation Rate in mm, beat_population, Temperature_C, Wind speed(km/h), TIME_OF_DAY_Afternoon, TIME_OF_DAY_Evening, TIME_OF_DAY_Morning, TIME_OF_DAY_Night, LIGHTING_CONDITION_DARKNESS, LIGHTING_CONDITION_DARKNESS, LIGHTED ROAD, LIGHTING_CONDITION_DAWN, LIGHTING_CONDITION_DAYLIGHT, LIGHTING_CONDITION_DUSK, LIGHTING_CONDITION_UNKNOWN",1.101669,1.104611,0.673157,0.670252,0.048866,0.044635


In [61]:
#order 2
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import RidgeCV

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)


alphas = np.logspace(-3, 3, 100)
ridge = RidgeCV(alphas=alphas, cv=10)
ridge.fit(X_train_scaled, y_train)


train_pred = ridge.predict(X_train_scaled)
test_pred = ridge.predict(X_test_scaled)

print(f"Best Alpha: {ridge.alpha_:.4f}")
print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, train_pred)):.4f}")
print(f"Test  RMSE: {np.sqrt(mean_squared_error(y_test, test_pred)):.4f}")
print(f"Train R²:   {r2_score(y_train, train_pred):.4f}")
print(f"Test  R²:   {r2_score(y_test, test_pred):.4f}")

# Top 3 most important predictors by absolute coefficient
feature_names = poly.get_feature_names_out(X_train.columns)
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': ridge.coef_
})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
top3 = coef_df.nlargest(3, 'Abs_Coefficient')[['Feature', 'Coefficient']].reset_index(drop=True)
top3.index += 1
top3

Best Alpha: 497.7024
Train RMSE: 1.0958
Test  RMSE: 1.0986
Train R²:   0.0590
Test  R²:   0.0551


,Feature,Coefficient
1,#_of_vehicles POSTED_SPEED_LIMIT,0.308838
2,#_of_vehicles TIME_OF_DAY_Night,-0.169075
3,#_of_vehicles TIME_OF_DAY_Afternoon,0.101684


In [64]:
print(f"Train MAE: {mean_absolute_error(y_train, train_pred):.4f}")
print(f"Test  MAE: {mean_absolute_error(y_test, test_pred):.4f}")


Train MAE: 0.6696
Test  MAE: 0.6666


In [88]:
coef_df.sort_values(by=['Abs_Coefficient']).tail(15)

,Feature,Coefficient,Abs_Coefficient
56,"POSTED_SPEED_LIMIT LIGHTING_CONDITION_DARKNESS, LIGHTED ROAD",0.036904,0.036904
126,"TIME_OF_DAY_Night LIGHTING_CONDITION_DARKNESS, LIGHTED ROAD",0.037105,0.037105
33,beat_population POSTED_SPEED_LIMIT,-0.037588,0.037588
51,POSTED_SPEED_LIMIT TIME_OF_DAY_Afternoon,-0.045446,0.045446
23,#_of_vehicles TIME_OF_DAY_Evening,0.045641,0.045641
54,POSTED_SPEED_LIMIT TIME_OF_DAY_Night,0.059912,0.059912
60,POSTED_SPEED_LIMIT LIGHTING_CONDITION_UNKNOWN,-0.063209,0.063209
29,#_of_vehicles LIGHTING_CONDITION_DAYLIGHT,0.071624,0.071624
17,#_of_vehicles beat_population,-0.079603,0.079603
31,#_of_vehicles LIGHTING_CONDITION_UNKNOWN,-0.083986,0.083986


## 6) Inference

In [101]:
train = X_train.copy()
train['INJURY_SCORE'] = y_train.values
train = train.rename(columns = {'#_of_vehicles':'num_of_vehicles'})

In [111]:
import statsmodels.formula.api as smf

model = smf.ols(formula = "INJURY_SCORE ~ POSTED_SPEED_LIMIT + TIME_OF_DAY_Afternoon + TIME_OF_DAY_Night + num_of_vehicles \
+ LIGHTING_CONDITION_UNKNOWN + LIGHTING_CONDITION_DAYLIGHT+ beat_population \
+ num_of_vehicles:POSTED_SPEED_LIMIT + I(beat_population**2) + num_of_vehicles:TIME_OF_DAY_Night + \
num_of_vehicles:TIME_OF_DAY_Afternoon + num_of_vehicles:beat_population + \
num_of_vehicles:LIGHTING_CONDITION_UNKNOWN + POSTED_SPEED_LIMIT:LIGHTING_CONDITION_UNKNOWN + \
POSTED_SPEED_LIMIT:TIME_OF_DAY_Night", data=train).fit()


In [112]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           INJURY_SCORE   R-squared:                       0.055
Model:                            OLS   Adj. R-squared:                  0.055
Method:                 Least Squares   F-statistic:                     3456.
Date:                Mon, 16 Mar 2026   Prob (F-statistic):               0.00
Time:                        22:49:40   Log-Likelihood:            -1.3547e+06
No. Observations:              895590   AIC:                         2.710e+06
Df Residuals:                  895574   BIC:                         2.710e+06
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
=========================================================================================================================
                                                            coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------------
Intercept                                                 0.9614      0.028     34.189      0.000       0.906       1.017
TIME_OF_DAY_Afternoon[T.True]                            -0.1873      0.013    -14.121      0.000      -0.213      -0.161
TIME_OF_DAY_Night[T.True]                                 0.2195      0.021     10.465      0.000       0.178       0.261
LIGHTING_CONDITION_UNKNOWN[T.True]                        0.2238      0.040      5.535      0.000       0.145       0.303
LIGHTING_CONDITION_DAYLIGHT[T.True]                      -0.0785      0.003    -25.042      0.000      -0.085      -0.072
POSTED_SPEED_LIMIT                                       -0.0176      0.001    -20.608      0.000      -0.019      -0.016
POSTED_SPEED_LIMIT:LIGHTING_CONDITION_UNKNOWN[T.True]    -0.0088      0.001    -10.921      0.000      -0.010      -0.007
POSTED_SPEED_LIMIT:TIME_OF_DAY_Night[T.True]              0.0130      0.001     22.882      0.000       0.012       0.014
num_of_vehicles                                           0.0589      0.014      4.305      0.000       0.032       0.086
num_of_vehicles:TIME_OF_DAY_Night[T.True]                -0.3076      0.006    -49.907      0.000      -0.320      -0.295
num_of_vehicles:TIME_OF_DAY_Afternoon[T.True]             0.1120      0.006     17.498      0.000       0.099       0.125
num_of_vehicles:LIGHTING_CONDITION_UNKNOWN[T.True]       -0.2130      0.017    -12.330      0.000      -0.247      -0.179
beat_population                                       -1.181e-05   1.19e-06     -9.893      0.000   -1.42e-05   -9.47e-06
num_of_vehicles:POSTED_SPEED_LIMIT                        0.0175      0.000     41.648      0.000       0.017       0.018
I(beat_population ** 2)                                6.393e-10   2.97e-11     21.547      0.000    5.81e-10    6.98e-10
num_of_vehicles:beat_population                        -6.64e-06   4.38e-07    -15.175      0.000    -7.5e-06   -5.78e-06
==============================================================================
Omnibus:                   878763.666   Durbin-Watson:                   2.000
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         97936738.074
Skew:                           4.550   Prob(JB):                         0.00
Kurtosis:                      53.415   Cond. No.                     8.77e+09
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 8.77e+09. This might indicate that there are
strong multicollinearity or other numerical problems.
"""